In [ ]:
import sys
!{sys.executable} -m pip install -q pandas==2.2.3 requests==2.32.3 beautifulsoup4==4.12.3 rapidfuzz==3.9.7 tqdm==4.66.5 duckduckgo-search==6.3.7 feedparser==6.0.11

In [ ]:
import os,re,time,random,hashlib,unicodedata
from datetime import datetime, timezone
import pandas as pd
from rapidfuzz import fuzz

def remove_accents(text):
    text='' if text is None else str(text)
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))
def normalize_text(text):
    t=remove_accents(text).lower(); t=re.sub(r'[^a-z0-9\s]',' ',t)
    return re.sub(r'\s+',' ',t).strip()
def normalize_name(name): return normalize_text(name)
def stable_hash(text): return hashlib.sha256(str(text).encode('utf-8')).hexdigest()[:16]
def score_name_match(a,b): return float(fuzz.token_sort_ratio(normalize_name(a),normalize_name(b)))
def classify_match(score):
    return 'exact_or_very_strong_name' if score>=95 else 'strong_candidate' if score>=88 else 'medium_candidate' if score>=75 else 'weak_candidate' if score>=60 else 'discard'
def now_utc(): return datetime.now(timezone.utc).isoformat()
def sha256_file(path):
    if not path or not os.path.exists(path): return ''
    h=hashlib.sha256();
    with open(path,'rb') as f:
        for ch in iter(lambda:f.read(8192),b''): h.update(ch)
    return h.hexdigest()
def save_csv(df,path): os.makedirs(os.path.dirname(path),exist_ok=True); df.to_csv(path,index=False,encoding='utf-8-sig')
EVID_COLS=['seed_id', 'nombre_persona_input', 'normalized_name_input', 'estado_input', 'puesto_input', 'dependencia_input', 'notas_input', 'source', 'source_type', 'source_country', 'source_official', 'source_category', 'source_reliability', 'matched_record_name', 'matched_record_normalized_name', 'matched_alias', 'matched_entity_type', 'matched_role', 'matched_position', 'matched_agency', 'matched_state', 'matched_country', 'matched_identifier', 'matched_company', 'match_score_pct', 'match_strength', 'match_method', 'matched_fields', 'conflicting_fields', 'requires_review', 'identity_confidence_pct', 'signal_type', 'signal_category', 'severity', 'risk_layer', 'risk_level_candidate', 'confidence_pct', 'evidence_title', 'evidence_summary', 'evidence_snippet', 'evidence_keywords', 'evidence_date', 'evidence_url', 'source_url', 'search_query', 'search_engine', 'retrieved_at', 'raw_file_path', 'raw_file_sha256', 'involvement_summary', 'explanation', 'limitations', 'recommended_action']

In [ ]:

import requests
from duckduckgo_search import DDGS
MAX_PEOPLE=10; MAX_RESULTS_PER_QUERY=5; ENABLE_DUCKDUCKGO=True; ENABLE_GDELT=True; SLEEP_MIN_SECONDS=3; SLEEP_MAX_SECONDS=7
kw=['corrupción','lavado','narcotráfico','enriquecimiento ilícito','sanción','investigación','FGR','OFAC','contratista','desvío de recursos','cohecho','peculado']
peps=pd.read_csv('notebooks/output/00_peps_normalized.csv',dtype=str).fillna('').head(MAX_PEOPLE)
ddg_rows=[]; gd_rows=[]; bench=[]
def mk(p,title,url,sn,src,stype,query):
    sc=score_name_match(p['nombre_persona_input'],title)
    return {**{c:'' for c in EVID_COLS},'seed_id':p['seed_id'],'nombre_persona_input':p['nombre_persona_input'],'normalized_name_input':p['normalized_name_input'],'estado_input':p.get('estado_input',''),'puesto_input':p.get('puesto_input',''),'dependencia_input':p.get('dependencia_input',''),'notas_input':p.get('notas_input',''),'source':src,'source_type':stype,'source_country':'INT','source_official':False,'source_category':'media_search','source_reliability':'secondary','matched_record_name':title,'matched_record_normalized_name':normalize_name(title),'match_score_pct':sc,'match_strength':classify_match(sc),'requires_review':True,'identity_confidence_pct':min(sc,75),'signal_type':'search_result' if stype=='search_engine' else 'gdelt_media_result','signal_category':'adverse_media','severity':'medium','risk_layer':'media','risk_level_candidate':'low_signal','confidence_pct':min(sc,75),'evidence_title':title,'evidence_summary':'Resultado localizado; resumen limitado al snippet; requiere revisión en fuente primaria.','evidence_snippet':sn,'evidence_url':url,'source_url':url,'search_query':query,'search_engine':'duckduckgo' if stype=='search_engine' else 'gdelt','retrieved_at':now_utc(),'involvement_summary':'URL localizada por buscador; requiere verificación en fuente primaria.','recommended_action':'verify_primary_source'}
if ENABLE_DUCKDUCKGO:
    t0=time.perf_counter();
    try:
        with DDGS() as ddgs:
            for _,p in peps.iterrows():
                for q in [f""{p['nombre_persona_input']}" {k}" for k in kw]:
                    for r in ddgs.text(q,max_results=MAX_RESULTS_PER_QUERY):
                        ddg_rows.append(mk(p,r.get('title',''),r.get('href',''),r.get('body',''),'DUCKDUCKGO','search_engine',q))
                    time.sleep(random.uniform(SLEEP_MIN_SECONDS,SLEEP_MAX_SECONDS))
        bench.append({'source':'DUCKDUCKGO','source_type':'search_engine','attempted':True,'success':True,'rows_downloaded_or_loaded':len(ddg_rows),'rows_parsed':len(ddg_rows),'matches_found':len(ddg_rows),'unique_people_with_hits':len(set([r['seed_id'] for r in ddg_rows])),'runtime_seconds':time.perf_counter()-t0,'error_message':'','notes':'Secondary signal'})
    except Exception as e:
        bench.append({'source':'DUCKDUCKGO','source_type':'search_engine','attempted':True,'success':False,'rows_downloaded_or_loaded':0,'rows_parsed':0,'matches_found':0,'unique_people_with_hits':0,'runtime_seconds':time.perf_counter()-t0,'error_message':str(e),'notes':'Controlled failure'})
if ENABLE_GDELT:
    t0=time.perf_counter()
    for _,p in peps.iterrows():
        q=f'"{p["nombre_persona_input"]}"'
        try:
            u=f'https://api.gdeltproject.org/api/v2/doc/doc?query={q}&mode=ArtList&maxrecords={MAX_RESULTS_PER_QUERY}&format=json'
            r=requests.get(u,timeout=45)
            if r.status_code==200:
                arts=r.json().get('articles',[])
                for a in arts: gd_rows.append(mk(p,a.get('title',''),a.get('url',''),a.get('seendate',''),'GDELT','media_api',q))
            time.sleep(random.uniform(SLEEP_MIN_SECONDS,SLEEP_MAX_SECONDS))
        except Exception:
            pass
    bench.append({'source':'GDELT','source_type':'media_api','attempted':True,'success':len(gd_rows)>0,'rows_downloaded_or_loaded':len(gd_rows),'rows_parsed':len(gd_rows),'matches_found':len(gd_rows),'unique_people_with_hits':len(set([r['seed_id'] for r in gd_rows])),'runtime_seconds':time.perf_counter()-t0,'error_message':'' if gd_rows else 'No results or request failure','notes':'Secondary signal'})
save_csv(pd.DataFrame(ddg_rows,columns=EVID_COLS),'notebooks/output/03_duckduckgo_evidence.csv')
save_csv(pd.DataFrame(gd_rows,columns=EVID_COLS),'notebooks/output/03_gdelt_evidence.csv')
allr=ddg_rows+gd_rows; ev=pd.DataFrame(allr,columns=EVID_COLS); save_csv(ev,'notebooks/output/03_adverse_media_evidence.csv')
ps=ev.groupby(['seed_id','nombre_persona_input'],as_index=False).size().rename(columns={'size':'total_signals'}) if len(ev) else pd.DataFrame(columns=['seed_id','nombre_persona_input','total_signals'])
save_csv(ps,'notebooks/output/03_adverse_media_person_summary.csv'); save_csv(pd.DataFrame(bench),'notebooks/output/03_benchmark_adverse_media.csv')
